# 中间件（Middleware）

Middleware 让你在 Agent 执行的各个环节插入自定义逻辑——重试、降级、缓存、内容过滤等，而不需要修改 Agent 本身的代码。

## 六个钩子点

| 钩子 | 执行频率 | 位置 | 用途 |
| --- | --- | --- | --- |
| before_agent | 1次 | Agent 开始前 | 初始化、权限检查 |
| before_model | 每次循环 | 模型调用前 | 消息预处理 |
| wrap_model_call | 每次循环 | 包裹模型调用 | 重试、降级、缓存 |
| after_model | 每次循环 | 模型调用后 | 内容审核、日志 |
| wrap_tool_call | 每次工具调用 | 包裹工具执行 | 工具重试、缓存 |
| after_agent | 1次 | Agent 结束后 | 格式化输出、统计 |

### 方式 1：装饰器（推荐）

In [1]:
from langchain.agents.middleware import before_model, after_model

@before_model
def log_before(state, runtime):
    """在每次模型调用前记录日志"""
    msg_count = len(state.get("messages", []))
    print(f"[before_model] 当前消息数: {msg_count}")
    return None

@after_model
def log_after(state, runtime):
    """在每次模型调用后记录日志"""
    last_msg = state["messages"][-1] if state.get("messages") else None
    if last_msg and hasattr(last_msg, 'tool_calls') and last_msg.tool_calls:
        print(f"[after_model] 模型请求了工具调用")
    return None

print("装饰器方式中间件已定义")

装饰器方式中间件已定义


### 方式 2：类继承（适合复杂逻辑）

In [2]:
from langchain.agents.middleware import AgentMiddleware

class LoggingMiddleware(AgentMiddleware):
    """自定义日志中间件"""

    @property
    def name(self) -> str:
        return "logging"

    def before_agent(self, state, runtime):
        print("[Logging] Agent 开始执行")
        return None

    def before_model(self, state, runtime):
        msg_count = len(state.get("messages", []))
        print(f"[Logging] 准备调用模型，当前 {msg_count} 条消息")
        return None

    def after_model(self, state, runtime):
        print("[Logging] 模型调用完成")
        return None

    def after_agent(self, state, runtime):
        print("[Logging] Agent 执行结束")
        return None

print("类继承方式中间件已定义")

类继承方式中间件已定义


## 完整的生命周期示例

In [4]:
from dotenv import load_dotenv
load_dotenv()

from langchain.agents import create_agent
from langchain.agents.middleware import before_agent, after_agent, before_model, after_model
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage
from langchain.tools import tool

@before_agent
def start_log(state, runtime):
    print(">>> [before_agent] Agent 开始 <<<")
    return None

@before_model
def pre_model(state, runtime):
    msg_count = len(state.get("messages", []))
    print(f" -> [before_model] 第 {msg_count} 条消息")
    return None

@after_model
def post_model(state, runtime):
    last = state["messages"][-1] if state.get("messages") else None
    if hasattr(last, 'tool_calls') and last.tool_calls:
        print(f" <- [after_model] 请求工具: {[tc['name'] for tc in last.tool_calls]}")
    else:
        content = str(last.content)[:50] if last and hasattr(last, 'content') else ""
        print(f" <- [after_model] 直接回复: {content}...")
    return None

@after_agent
def end_log(state, runtime):
    total = len(state.get("messages", []))
    print(f"<<< [after_agent] Agent 结束，共 {total} 条消息 <<<")
    return None

@tool
def get_weather(city: str) -> str:
    """查询天气"""
    return f"{city}: 晴，25°C"

model = init_chat_model("deepseek:deepseek-v4-flash", temperature=0)
agent = create_agent(
    model=model, tools=[get_weather],
    middleware=[start_log, pre_model, post_model, end_log],
    system_prompt="你是助手。",
)

result = agent.invoke({"messages": [HumanMessage(content="杭州天气？")]})
print(f"\n最终回复: {result['messages'][-1].content}")

>>> [before_agent] Agent 开始 <<<
 -> [before_model] 第 1 条消息
 <- [after_model] 请求工具: ['get_weather']
 -> [before_model] 第 3 条消息
 <- [after_model] 直接回复: 杭州今天天气晴朗，气温 **25°C**，是个不错的好天气！☀️

有什么其他需要帮忙的吗？😊...
<<< [after_agent] Agent 结束，共 4 条消息 <<<

最终回复: 杭州今天天气晴朗，气温 **25°C**，是个不错的好天气！☀️

有什么其他需要帮忙的吗？😊


## Middleware 的返回值

| 返回值 | 效果 |
| --- | --- |
| None | 不修改状态，继续流程 |
| dict | 更新 Agent 状态（合并到当前状态） |
| 含 jump_to 的 dict | 跳转到指定节点 |